# 02 Model Training
## LSTM Model Development and Training

This notebook trains an LSTM model for stock price prediction.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.data_loader import download_stock_data, get_data_stats
from src.preprocessing import DataPreprocessor, create_dataset, train_test_split, reshape_for_lstm
from src.model import LSTMModel

print("Libraries imported successfully!")

## Step 1: Configuration

In [ ]:
# Configuration
ticker = 'AAPL'
start_date = '2020-01-01'
end_date = '2026-02-01'

# Model hyperparameters
time_steps = 60  # Use 60 days to predict next day
lstm_units = 50
num_layers = 3
dropout_rate = 0.2
epochs = 50
batch_size = 32

print(f"Configuration:")
print(f"  Ticker: {ticker}")
print(f"  Time Period: {start_date} to {end_date}")
print(f"  Time Steps: {time_steps}")
print(f"  LSTM Units: {lstm_units}")
print(f"  Number of Layers: {num_layers}")
print(f"  Dropout Rate: {dropout_rate}")
print(f"  Epochs: {epochs}")
print(f"  Batch Size: {batch_size}")

## Step 2: Download and Load Data

In [ ]:
print("Downloading stock data...")
data = download_stock_data(ticker, start_date, end_date)
get_data_stats(data, ticker)

## Step 3: Data Preprocessing

In [ ]:
print("\nPreprocessing data...")
preprocessor = DataPreprocessor()

# Handle missing values
data = preprocessor.handle_missing_values(data, method='forward_fill')

# Remove outliers
data_clean = preprocessor.remove_outliers(data, column='Close', std_multiplier=3)

# Scale data
scaled_data = preprocessor.scale_data(data_clean[['Close']])

print(f"\nCleaned data shape: {data_clean.shape}")
print(f"Scaled data shape: {scaled_data.shape}")

## Step 4: Create Training Dataset

In [ ]:
print("\nCreating training dataset...")
X, y = create_dataset(scaled_data, time_step=time_steps)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Reshape for LSTM
print("\nReshaping for LSTM...")
X_train_reshaped = reshape_for_lstm(X_train)
X_test_reshaped = reshape_for_lstm(X_test)

print(f"\nTrain data shape: {X_train_reshaped.shape}")
print(f"Test data shape: {X_test_reshaped.shape}")

## Step 5: Build LSTM Model

In [ ]:
print("\nBuilding LSTM model...")
model = LSTMModel(
    time_steps=time_steps,
    units=lstm_units,
    dropout_rate=dropout_rate,
    num_layers=num_layers
)

model.build_model()

## Step 6: Train Model

In [ ]:
print("\nTraining model...")
history = model.train(
    X_train_reshaped, y_train,
    X_test_reshaped, y_test,
    epochs=epochs,
    batch_size=batch_size,
    early_stoppage=True
)

## Step 7: Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE plot
axes[1].plot(history.history['mean_absolute_error'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mean_absolute_error'], label='Validation MAE', linewidth=2)
axes[1].set_title('Mean Absolute Error', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Model Prediction and Evaluation

In [ ]:
# Evaluate model
print("Evaluating model...")
metrics = model.evaluate(X_test_reshaped, y_test)

# Make predictions
y_pred_train = model.predict(X_train_reshaped)
y_pred_test = model.predict(X_test_reshaped)

# Inverse transform
y_train_actual = preprocessor.inverse_transform(y_train.reshape(-1, 1))
y_test_actual = preprocessor.inverse_transform(y_test.reshape(-1, 1))
y_pred_train_actual = preprocessor.inverse_transform(y_pred_train)
y_pred_test_actual = preprocessor.inverse_transform(y_pred_test)

print(f"Predictions ready for evaluation")

## Step 9: Calculate Detailed Metrics

In [ ]:
# Training metrics
train_rmse = np.sqrt(mean_squared_error(y_train_actual, y_pred_train_actual))
train_mae = mean_absolute_error(y_train_actual, y_pred_train_actual)
train_r2 = r2_score(y_train_actual, y_pred_train_actual)

# Test metrics
test_rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_test_actual))
test_mae = mean_absolute_error(y_test_actual, y_pred_test_actual)
test_r2 = r2_score(y_test_actual, y_pred_test_actual)

# Directional accuracy
direction_actual_test = np.diff(y_test_actual.flatten()) > 0
direction_pred_test = np.diff(y_pred_test_actual.flatten()) > 0
directional_accuracy = np.mean(direction_actual_test == direction_pred_test) * 100

print(f"\n{'='*70}")
print("MODEL PERFORMANCE METRICS")
print(f"{'='*70}")
print(f"\nTRAINING SET:")
print(f"  RMSE: ${train_rmse:.2f}")
print(f"  MAE: ${train_mae:.2f}")
print(f"  R²: {train_r2:.4f}")

print(f"\nTEST SET:")
print(f"  RMSE: ${test_rmse:.2f}")
print(f"  MAE: ${test_mae:.2f}")
print(f"  R²: {test_r2:.4f}")
print(f"  Directional Accuracy: {directional_accuracy:.2f}%")
print(f"{'='*70}")

## Step 10: Visualize Predictions

In [ ]:
# Training predictions
plt.figure(figsize=(14, 6))
plt.plot(y_train_actual, label='Actual Price', color='blue', linewidth=2, alpha=0.7)
plt.plot(y_pred_train_actual, label='Predicted Price', color='red', linewidth=2, alpha=0.7)
plt.title(f'{ticker} - Training Set Predictions', fontsize=14, fontweight='bold')
plt.xlabel('Time Period', fontsize=12)
plt.ylabel('Stock Price ($)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Test predictions
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual, label='Actual Price', color='blue', linewidth=2, alpha=0.7)
plt.plot(y_pred_test_actual, label='Predicted Price', color='red', linewidth=2, alpha=0.7)
plt.title(f'{ticker} - Test Set Predictions', fontsize=14, fontweight='bold')
plt.xlabel('Time Period', fontsize=12)
plt.ylabel('Stock Price ($)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 11: Residual Analysis

In [ ]:
# Calculate residuals
residuals_test = y_test_actual.flatten() - y_pred_test_actual.flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals histogram
axes[0].hist(residuals_test, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Residual ($)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].axvline(0, color='red', linestyle='--', linewidth=2)
axes[0].grid(True, alpha=0.3, axis='y')

# Residuals over time
axes[1].plot(residuals_test, linewidth=1, alpha=0.7, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].fill_between(range(len(residuals_test)), residuals_test, 0, 
                     where=(residuals_test >= 0), alpha=0.3, color='green')
axes[1].fill_between(range(len(residuals_test)), residuals_test, 0, 
                     where=(residuals_test < 0), alpha=0.3, color='red')
axes[1].set_title('Residuals Over Time', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Residual ($)', fontsize=11)
axes[1].set_xlabel('Time Period', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residual Statistics:")
print(f"  Mean: ${residuals_test.mean():.2f}")
print(f"  Std Dev: ${residuals_test.std():.2f}")
print(f"  Min: ${residuals_test.min():.2f}")
print(f"  Max: ${residuals_test.max():.2f}")

## Step 12: Save Model

In [ ]:
# Create directories if needed
os.makedirs('../models/saved_models', exist_ok=True)

# Save model and config
model_path = f'../models/saved_models/{ticker}_lstm_model.h5'
config_path = f'../models/saved_models/{ticker}_config.json'

model.save_model(model_path)
model.save_config(config_path)

print(f"\nModel successfully saved!")
print(f"Ready for predictions!")